# EDA: Opportunity Score Validation

Purpose: validate the final opportunity mart for score distributions, tier balance, high-opportunity markets, and interpretation before Tableau publishing.

## Setup

Install if needed:

```bash
pip install pandas numpy matplotlib snowflake-connector-python
```

In [ ]:

from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

DATABASE = "SMB_MARKET_INTELLIGENCE_DEV"
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH")
ROLE = os.getenv("SNOWFLAKE_ROLE", "ACCOUNTADMIN")

def get_connection():
    """Create a Snowflake connection from environment variables.

    Required environment variables for password auth:
        SNOWFLAKE_ACCOUNT
        SNOWFLAKE_USER
        SNOWFLAKE_PASSWORD

    Optional:
        SNOWFLAKE_AUTHENTICATOR=externalbrowser
        SNOWFLAKE_WAREHOUSE=COMPUTE_WH
        SNOWFLAKE_ROLE=ACCOUNTADMIN
    """
    import snowflake.connector

    account = os.getenv("SNOWFLAKE_ACCOUNT")
    user = os.getenv("SNOWFLAKE_USER")
    password = os.getenv("SNOWFLAKE_PASSWORD")
    authenticator = os.getenv("SNOWFLAKE_AUTHENTICATOR")

    if not account or not user:
        raise RuntimeError(
            "Set SNOWFLAKE_ACCOUNT and SNOWFLAKE_USER before running this notebook. "
            "For browser auth, also set SNOWFLAKE_AUTHENTICATOR=externalbrowser. "
            "For password auth, set SNOWFLAKE_PASSWORD."
        )

    kwargs = {
        "account": account,
        "user": user,
        "warehouse": WAREHOUSE,
        "database": DATABASE,
        "role": ROLE,
    }

    if authenticator:
        kwargs["authenticator"] = authenticator
    elif password:
        kwargs["password"] = password
    else:
        raise RuntimeError(
            "Set either SNOWFLAKE_PASSWORD or SNOWFLAKE_AUTHENTICATOR=externalbrowser."
        )

    return snowflake.connector.connect(**kwargs)

def read_sql(query):
    with get_connection() as conn:
        return pd.read_sql(query, conn)

def show_basic_profile(df):
    display(df.head())
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")


## Load final mart

In [ ]:

mart = read_sql("""
SELECT
    scoring_version,
    scored_at,
    state_fips,
    state_abbr,
    state_name,
    county_fips,
    qcew_industry_code,
    sector_name,
    year,
    quarter,
    period_id,
    qtrly_estabs,
    avg_monthly_employment,
    total_qtrly_wages,
    ongoing_growth_score,
    expected_growth_score,
    ongoing_growth_tier,
    expected_growth_tier,
    sba_loan_count,
    sba_gross_approval_amount,
    active_lender_count,
    top_lender_name,
    top_lender_share_by_amount,
    lender_hhi_by_amount,
    lender_concentration_tier,
    sba_loans_per_100_establishments,
    sba_dollars_per_establishment,
    lending_penetration_score,
    low_lending_penetration_score,
    underserved_score,
    lender_fragmentation_score,
    lending_momentum_score,
    final_opportunity_score,
    opportunity_tier,
    recommendation
FROM MART.COUNTY_INDUSTRY_OPPORTUNITY_QTR
""")

mart.columns = mart.columns.str.lower()
show_basic_profile(mart)


## DQ current status

In [ ]:

dq = read_sql("""
SELECT
    check_group,
    check_name,
    severity,
    object_name,
    status,
    observed_value,
    expected_value,
    details,
    checked_at
FROM DQ.VW_DQ_CURRENT_STATUS
ORDER BY
    CASE status WHEN 'FAIL' THEN 1 ELSE 2 END,
    CASE severity WHEN 'ERROR' THEN 1 WHEN 'WARN' THEN 2 ELSE 3 END,
    check_group,
    check_name
""")

dq.columns = dq.columns.str.lower()
display(dq)


## Score distributions

In [ ]:

score_cols = [
    "ongoing_growth_score",
    "expected_growth_score",
    "lending_penetration_score",
    "low_lending_penetration_score",
    "underserved_score",
    "lender_fragmentation_score",
    "lending_momentum_score",
    "final_opportunity_score",
]

display(mart[score_cols].describe().T)


In [ ]:

for score_col in score_cols:
    plt.figure(figsize=(8, 4.5))
    plt.hist(mart[score_col].dropna(), bins=30)
    plt.title(f"{score_col} distribution")
    plt.xlabel(score_col)
    plt.ylabel("Rows")
    plt.tight_layout()
    plt.show()


## Tier balance

In [ ]:

tier_summary = (
    mart.groupby("opportunity_tier")
    .agg(
        rows=("county_fips", "size"),
        avg_final_score=("final_opportunity_score", "mean"),
        avg_underserved_score=("underserved_score", "mean"),
        avg_ongoing_growth_score=("ongoing_growth_score", "mean"),
        avg_expected_growth_score=("expected_growth_score", "mean"),
        avg_sba_loans_per_100_estabs=("sba_loans_per_100_establishments", "mean"),
        avg_sba_dollars_per_estab=("sba_dollars_per_establishment", "mean"),
    )
    .reset_index()
    .sort_values("avg_final_score", ascending=False)
)

display(tier_summary)


In [ ]:

plt.figure(figsize=(8, 4.5))
plt.bar(tier_summary["opportunity_tier"], tier_summary["rows"])
plt.title("Opportunity rows by tier")
plt.xlabel("Opportunity tier")
plt.ylabel("Rows")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Latest-period top opportunities

In [ ]:

latest_period = mart["period_id"].max()
latest = mart[mart["period_id"] == latest_period].copy()

top_latest = (
    latest.sort_values("final_opportunity_score", ascending=False)
    .loc[
        :,
        [
            "state_abbr",
            "county_fips",
            "sector_name",
            "qtrly_estabs",
            "ongoing_growth_score",
            "expected_growth_score",
            "underserved_score",
            "lender_fragmentation_score",
            "final_opportunity_score",
            "opportunity_tier",
            "sba_loan_count",
            "sba_gross_approval_amount",
            "sba_loans_per_100_establishments",
            "top_lender_name",
            "recommendation",
        ],
    ]
    .head(50)
)

display(top_latest)


## Component correlation check

This checks whether the final score is behaving as expected. It should correlate strongly with growth and underserved components, but not be a disguised copy of any single component.

In [ ]:

corr = mart[score_cols].corr(numeric_only=True)
display(corr)


In [ ]:

component_cols = [
    "ongoing_growth_score",
    "expected_growth_score",
    "underserved_score",
    "lender_fragmentation_score",
    "lending_momentum_score",
]

for col in component_cols:
    plt.figure(figsize=(6, 4.5))
    plt.scatter(mart[col], mart["final_opportunity_score"], alpha=0.2)
    plt.title(f"Final opportunity score vs {col}")
    plt.xlabel(col)
    plt.ylabel("final_opportunity_score")
    plt.tight_layout()
    plt.show()


## State and sector rankings

In [ ]:

state_rank = (
    latest.groupby(["state_abbr", "state_name"])
    .agg(
        rows=("county_fips", "size"),
        high_or_better_rows=("opportunity_tier", lambda s: s.isin(["Very High", "High"]).sum()),
        avg_final_score=("final_opportunity_score", "mean"),
        avg_underserved_score=("underserved_score", "mean"),
        total_establishments=("qtrly_estabs", "sum"),
        total_sba_loans=("sba_loan_count", "sum"),
        total_sba_amount=("sba_gross_approval_amount", "sum"),
    )
    .reset_index()
    .sort_values(["high_or_better_rows", "avg_final_score"], ascending=False)
)

display(state_rank.head(25))


In [ ]:

sector_rank = (
    latest.groupby(["qcew_industry_code", "sector_name"])
    .agg(
        rows=("county_fips", "size"),
        high_or_better_rows=("opportunity_tier", lambda s: s.isin(["Very High", "High"]).sum()),
        avg_final_score=("final_opportunity_score", "mean"),
        avg_underserved_score=("underserved_score", "mean"),
        total_establishments=("qtrly_estabs", "sum"),
        total_sba_loans=("sba_loan_count", "sum"),
        total_sba_amount=("sba_gross_approval_amount", "sum"),
    )
    .reset_index()
    .sort_values(["high_or_better_rows", "avg_final_score"], ascending=False)
)

display(sector_rank)


## Notes to capture

- Are high-opportunity rows concentrated in plausible sectors/states?
- Does the final score reward growth while also capturing low SBA penetration?
- Are no-SBA markets preserved and interpretable?
- Should tier thresholds be adjusted before Tableau publication?